In [1]:
import os
import zipfile
import shutil
import random

zip_filename = "archive.zip"
extract_path = "./lfw_dataset"
base_roles_path = "./roles_data"

roles = ["student", "staff", "management"]

# 1. Unzip dataset if archive.zip exists and folder is missing
if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
    print(f"Dataset already extracted in '{extract_path}'.")
elif os.path.exists(zip_filename):
    print("Extracting archive.zip...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Handle inner zip files if packaged inside archive.zip
    for root, dirs, files_list in os.walk(extract_path):
        for f in files_list:
            if f.endswith('.zip'):
                with zipfile.ZipFile(os.path.join(root, f), 'r') as inner_ref:
                    inner_ref.extractall(extract_path)
    print("Dataset extracted successfully.")
else:
    print(f"Notice: '{zip_filename}' not found. Ensure 'archive.zip' is placed in the sidebar.")

# 2. Build role directories
for role in roles:
    os.makedirs(os.path.join(base_roles_path, role), exist_ok=True)

# 3. Collect images from extracted dataset
all_images = []
if os.path.exists(extract_path):
    for root, dirs, files_list in os.walk(extract_path):
        for f in files_list:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(root, f))

# 4. Populate role folders with specific explicit IDs as file names
random.shuffle(all_images)

id_prefixes = {
    "student": "STU_",
    "staff": "STF_",
    "management": "MGT_"
}

if len(all_images) >= 30:
    for idx, role in enumerate(roles):
        role_dir = os.path.join(base_roles_path, role)
        prefix = id_prefixes[role]

        # Populate only if empty to preserve manual files
        if len(os.listdir(role_dir)) == 0:
            sample_imgs = all_images[idx*10 : (idx+1)*10]
            for i, img_path in enumerate(sample_imgs):
                custom_id = f"{prefix}{(i+101):03d}"  # e.g. STU_101, STF_101, MGT_101
                file_extension = os.path.splitext(img_path)[1]
                new_file_name = f"{custom_id}{file_extension}"

                shutil.copy(img_path, os.path.join(role_dir, new_file_name))

print("\n---------------- ROLE FOLDER CREATION COMPLETE ----------------")
for role in roles:
    role_dir = os.path.join(base_roles_path, role)
    files = os.listdir(role_dir)
    print(f"Folder: {role_dir} ({len(files)} files)")
    if len(files) > 0:
        print(f" Sample IDs: {files[:3]}")
print("----------------------------------------------------------------")

Extracting archive.zip...
Dataset extracted successfully.

---------------- ROLE FOLDER CREATION COMPLETE ----------------
Folder: ./roles_data/student (10 files)
 Sample IDs: ['STU_106.jpg', 'STU_110.jpg', 'STU_109.jpg']
Folder: ./roles_data/staff (10 files)
 Sample IDs: ['STF_108.jpg', 'STF_104.jpg', 'STF_106.jpg']
Folder: ./roles_data/management (10 files)
 Sample IDs: ['MGT_106.jpg', 'MGT_102.jpg', 'MGT_104.jpg']
----------------------------------------------------------------


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomFaceNet(nn.Module):
    def __init__(self, embedding_size=128):
        super(CustomFaceNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, embedding_size)

    def forward_once(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, p=2, dim=1)

    def forward(self, anchor, positive, negative):
        return self.forward_once(anchor), self.forward_once(positive), self.forward_once(negative)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomFaceNet(embedding_size=128).to(device)
model.eval()

print("Custom FaceNet model loaded on device:", device)

Custom FaceNet model loaded on device: cpu


In [3]:
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
from datetime import datetime
import json

# Database storing registered identities: { "EXTRACTED_ID": {"role": role, "embedding": vector} }
registered_users = {}

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def register_directory_faces():
    """Reads image files, extracts IDs directly from file names, and generates embeddings."""
    registered_users.clear()
    for role in ["student", "staff", "management"]:
        role_path = os.path.join(base_roles_path, role)
        if not os.path.exists(role_path):
            continue

        for img_name in os.listdir(role_path):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                # File name (without extension) is treated as the official ID
                user_id = os.path.splitext(img_name)[0]
                img_path = os.path.join(role_path, img_name)

                img = Image.open(img_path).convert('RGB')
                tensor = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    embedding = model.forward_once(tensor).cpu().numpy().flatten()

                registered_users[user_id] = {
                    "role": role,
                    "embedding": embedding
                }

register_directory_faces()
print(f"Registered {len(registered_users)} profile face vectors directly from file IDs.")

def predict_face_category(img_path, threshold=0.65):
    """Matches photo against registered embeddings and returns backend JSON with IDs."""
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        live_embedding = model.forward_once(tensor).cpu().numpy().flatten()

    best_id = None
    best_role = "UNKNOWN"
    min_dist = float('inf')

    for uid, data in registered_users.items():
        dist = np.linalg.norm(live_embedding - data["embedding"])
        if dist < min_dist:
            min_dist = dist
            best_id = uid
            best_role = data["role"]

    is_match = min_dist <= threshold

    # Assign extracted ID to the specific matching role key
    student_id = best_id if (is_match and best_role == "student") else None
    staff_id = best_id if (is_match and best_role == "staff") else None
    management_id = best_id if (is_match and best_role == "management") else None

    payload = {
        "status": "SUCCESS" if is_match else "UNRECOGNIZED",
        "category": best_role if is_match else "UNKNOWN",
        "student_id": student_id,
        "staff_id": staff_id,
        "management_id": management_id,
        "distance": float(round(min_dist, 4)),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    print("\n---------------- RECOGNITION REPORT ----------------")
    print(f"Status        : {payload['status']}")
    print(f"Category      : {payload['category'].upper()}")
    print(f"Student ID    : {payload['student_id']}")
    print(f"Staff ID      : {payload['staff_id']}")
    print(f"Management ID : {payload['management_id']}")
    print(f"Distance Score: {payload['distance']}")
    print("----------------------------------------------------")

    return json.dumps(payload, indent=2)

Registered 30 profile face vectors directly from file IDs.


In [4]:
from google.colab import files

print("Select a photo to run inference on:")
uploaded = files.upload()

if uploaded:
    file_path = list(uploaded.keys())[0]

    # Run recognition pipeline
    json_result = predict_face_category(file_path)

    print("\nFormatted JSON Response for Backend Hand-off:")
    print(json_result)
else:
    print("No image file was uploaded.")

Select a photo to run inference on:


Saving MGT_104.jpg to MGT_104.jpg

---------------- RECOGNITION REPORT ----------------
Status        : SUCCESS
Category      : MANAGEMENT
Student ID    : None
Staff ID      : None
Management ID : MGT_104
Distance Score: 0.0
----------------------------------------------------

Formatted JSON Response for Backend Hand-off:
{
  "status": "SUCCESS",
  "category": "management",
  "student_id": null,
  "staff_id": null,
  "management_id": "MGT_104",
  "distance": 0.0,
  "timestamp": "2026-09-12 04:32:14"
}
